# Evaluating LLMs as Annotators

**Deep Learning in NLP**  
Groups 2–3 (RISS annotation scheme)  
23 June 2026

## General introduction

## Objectives
In this session, you will:

1. Load a human-annotated gold-standard dataset.
2. Formulate LLM inputs and generate annotations using an open-weight LLM.
3. Measure LLM–gold agreement.
4. Compare annotation quality across the following methods for an LLM:
   - zero-shot prompting;
   - few-shot prompting;
   - fine-tuning.

## Models

The notebook can be used with several open-weight models, including:

- `Qwen3-1.7B` <-- used as default
- `Qwen3.6-27B`
- `gemma-4-31b-it`
- `gpt-oss-120b`

## Expected Outcomes

By the end of the session, you should be able to:

- explain the difference between zero-shot, few-shot, and fine-tuned as LLM-annotation methods;
- inspect failure modes of LLM-based annotation methods across annotation categories;
- interpret evaluation metrics and compare methods performance.

---
**To keep the workflows manageable and self-contained, the session materials are split across separate notebooks for zero-shot, few-shot, and fine-tuning approaches.**

**Important:** Before making changes to this shared notebook, save your own copy:

`File → Save a copy in Drive`

**Check that you are running on GPU:**

`Runtime → Change runtime type → T4 GPU`

## LLM Annotation pipeline
(note differences between human and LLM procedure)

MAIN TASK: use an instruction-tuned large language model (LLM) as an automatic annotator and compare its predictions against the human gold standard.

```text
groups
 ├─ group_mentioned
 ├─ multiple_groups
 └─ first_group
        │
        ▼
group_appealed?
        │
        ├── no → stop
        │
        └── yes
              ├─ opposed_groups
              ├─ pejorative
              ├─ group_tags
              └─ appeal classification
                   ├─ reasoning
                   ├─ polarity
                   └─ stance
```                   


## Part 1. Methods: Zero-shot vs. Few-shot prompting

Large language models can be adapted to annotation tasks in several ways.

### Zero-shot prompting

The model receives:

* a task description,
* label definitions,
* annotation instructions,
* the text to annotate.

No annotated examples are provided in the prompt.

The model must infer how to apply the annotation scheme solely from the instructions.

### Few-shot prompting

The model receives:

* the task description,
* label definitions,
* several annotated examples,
* the text to annotate.

The examples demonstrate how the annotation scheme should be applied and serve as in-context training data.

---

The prompts are loaded from the GitHub repository. Inspect the prompts before running.

The resulting annotations are compared to the human gold standard using the same function as for human-human setup.

Don't forget to save the results externally.


In [27]:
# Download GitHub companion for this session to CoLab
from pathlib import Path
import os
import sys

REPO_DIR = Path("/content/DLDH-SocSci")

if not REPO_DIR.exists():
    !git clone https://github.com/kunilovskaya/DLDH-SocSci.git
else:
    print("Repository already present. Pulling latest changes...")
    %cd /content/DLDH-SocSci
    !git pull
    %cd /content

sys.path.insert(0, str(REPO_DIR))

# replace RUN with main_student_groups for the data from Groups2-3-on-riss-scheme (DL in NLP course) 15 June-30 June 2026
RUN = "trial_student_groups"  # main_student_groups
my_date = "16June2026"  # 29June2026 30June2026

GOLD_DIR = REPO_DIR / "data" / RUN

PROMPT_DIR = REPO_DIR / "prompts"

print("=" * 80)
print("Repository loaded from:", REPO_DIR)
print(f"Working on =={RUN}==")
print("=" * 80)


Repository already present. Pulling latest changes...
/content/DLDH-SocSci
Already up to date.
/content
Repository loaded from: /content/DLDH-SocSci
Working on ==trial_student_groups==


In [28]:
# Install required libraries
!pip install krippendorff

In [29]:
# load and explore gold dataset
import pandas as pd
from ast import literal_eval

df = pd.read_csv(GOLD_DIR / f"{my_date}_gold_dataset.tsv", sep="\t")

# pandas might read our "NA" as NaN
df["group_tags"] = df["group_tags"].apply(lambda x: literal_eval(x) if x else ["NA"])
df = df.fillna("NA")


df = df.sort_values("disagreement", ascending=False).reset_index(drop=True)
print(df[["sent_id", "disagreement", "group_tags", "text"]].head())
print(df.columns.tolist)

print("=" * 80)
print(f"Available data: {df.shape}")
print("=" * 80)

# are there any missing values in LLM annotations?
for annotator in ["gold"]:
    missing = df[
        (df["annotator"] == annotator) &
        df.isna().any(axis=1)
    ]
    if missing.empty:
        print(f"{annotator}: {len(missing)} rows with missing values")
    else:
        print(f"{annotator}: {len(missing)} rows with missing values")
        print(missing[["sent_id", "reasoning"]])

                  sent_id  disagreement  \
0  commons_2025_18596:002            12   
1  commons_1980_44470:053            10   
2  commons_2025_15818:005             8   
3  commons_2025_43572:018             8   
4  commons_2025_64095:011             7   

                                          group_tags  \
0                                        [AGE_CHILD]   
1                                   [INC_PRECARIOUS]   
2                    [OCC_AGRICULTURE, OTH_NON_RISS]   
3                    [OCC_AGRICULTURE, OTH_NON_RISS]   
4  [AGE_SCHOOL, EDU_IN_EDU, OCC_EDUCATORS, OTH_NO...   

                                                text  
0  On the past two occasions I have been in Ukrai...  
1  If it is transitional, it it not the fault of ...  
2                 Farmers are not multimillionaires.  
3  As part of our commitment to strengthening vit...  
4  There is a worrying decline in the recruitment...  
<bound method IndexOpsMixin.tolist of Index(['annotator', 'sent_id', 'upda

In [30]:
# get the sent_id with the lowest disagreement and detected appeal
my_row = df[df["group_appealed"] == "yes"].iloc[4]
# my_row = df[df["multiple_groups"] == "yes"].iloc[0]
sent_id = my_row["sent_id"]

print(my_row)

annotator                                                       gold
sent_id                                       commons_2025_64095:011
updated_at                                                2026-07-04
group_mentioned                                                  yes
group_appealed                                                   yes
intersectional                                                   yes
multiple_groups                                                   no
opposed_groups                                                    no
pejorative                                                        no
group_tags         [AGE_SCHOOL, EDU_IN_EDU, OCC_EDUCATORS, OTH_NO...
reasoning                                             REASONING_BOTH
polarity                                          POLARITY_PRO_GROUP
stance                                                STANCE_ENDORSE
text               There is a worrying decline in the recruitment...
left_context       The number of s

In [31]:
# Load model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [32]:
import torch

print("=" * 80)
if torch.cuda.is_available():
    print("Are GPUs connected? -- True --")
    print(f"You are on {torch.cuda.get_device_name(0)}")
else:
    print("Are GPUs connected? -- False --")
    print("You are on CPU")
print("=" * 80)

Are GPUs connected? -- True --
You are on Tesla T4


In [33]:
# helper functions for input and output processing
# define generic inference function
def ask_llm(prompt, max_new_tokens=200):

    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response

# define json parser to extracts just the JSON object from the response
# (the response might include noise) and
# to converts it into a Python dictionary
import json
import re

def parse_json(response):

    match = re.search(r"\{.*\}", response, re.DOTALL)

    if not match:
        return None

    return json.loads(match.group())

# === Experimenting with simpler prompts to find fomulations that work reliably ===

In [34]:
# do the same to test all prompts in the pipeline (see next cell)
from textwrap import fill
SIMPLE_EXTRACT  = """
List social groups mentioned in the TARGET sentence.
TARGET:
{text}

Return JSON:

{{
    "groups": [""]
}}
""".strip()

prompt = SIMPLE_EXTRACT.format(text=my_row['text'])
# prompt = SIMPLE_EXTRACT.format(text="Disabled people will receive additional support.")
response = ask_llm(prompt)
data = parse_json(response)  # returns Python dictionary
print("\n=== MENTIONS PROMPT ===")
print(prompt)
print("=" * 30 + "\n")

print("\n=== RAW RESPONSE TO MENTIONS RESULT ===")
print(response)
print("=" * 30)

print("\n=== MENTIONS RESULT ===")
print(data)
print("=" * 30 + "\n")


=== MENTIONS PROMPT ===
List social groups mentioned in the TARGET sentence.
TARGET:
There is a worrying decline in the recruitment and retention of music teachers.

Return JSON:

{
    "groups": [""]
}


=== RAW RESPONSE TO MENTIONS RESULT ===
{
    "groups": ["music teachers"]
}

=== MENTIONS RESULT ===
{'groups': ['music teachers']}



In [35]:
# define prompts: it is assumed that you tested them separately
MENTIONS_PROMPT = """
List social groups mentioned in the TARGET sentence.
{item}

Return JSON:

{{
    "groups": [""]
}}
""".strip()

OPPOSED_PROMPT = """
TASK:
Decide if the DETECTED social groups in the TARGET sentence are opposed or not.
Assign yes when groups are explicitly contrasted, compared, or set against each other.

Examples:

- immigrant men versus native women -> yes
- unlike pensioners, students... -> yes
- refugees receive support whereas veterans do not -> yes

Otherwise assign no.

DETECTED social groups:
{groups}

RULES:
- Annotate only the TARGET sentence.
- Use PREV and NEXT only to understand of the TARGET sentence.
- Replace <your_answer> with either "yes" or "no".

{item}

Return JSON:

{{
    "opposed_groups": "<your_answer>"
}}
""".strip()

PEJORATIVE_PROMPT = """
TASK:
Decide if the reference to the DETECTED group in the TARGET sentence is pejorative.

DEFINITION:
A pejorative reference is a derogatory, insulting, stigmatizing label or slur term.
Interpretation of regional labels depends on the context.

Examples:

- NAFRI instead of Nordafrikaner -> yes
- Welfare queen instead of welfare recipient -> yes
- Ossi for East Germans -> decide in context
- low-income families -> no

DETECTED group:
{group}

RULES:
- Annotate only the TARGET sentence.
- Use PREV and NEXT only to understand the TARGET sentence.
- Replace <your_answer> with either "yes" or "no".

{item}

Return JSON:

{{
    "pejorative": "<your_answer>"
}}
""".strip()

TAXONOMY_PROMPT = """
TASK: Assign one or more of the following categories to DETECTED group in TARGET sentence:

* age
* gender_sexuality
* family
* disability
* ethnicity
* migration_citizenship
* religion
* geography
* education
* income
* occupation
* other

## RULES
- Annotate only the DETECTED group in the TARGET sentence.
- Use PREV and NEXT only to understand the TARGET sentence.
- Assign all applicable categories. A group may belong to multiple categories simultaneously.
- Use "other" only when none of the other categories apply.
- Return only category labels from the list above.

Examples:

* immigrant women → migration_citizenship, gender_sexuality
* disabled children → disability, age
* elderly refugees → age, migration_citizenship
* working mothers → family, occupation

DETECTED group:
{group}

{item}

Return JSON:

{{
    "group_tags": [""]
}}
""".strip()

APPEAL_PROMPT = """
TASK:
Decide if the DETECTED group in the TARGET sentence is appealed to in a political argument.

DEFINITION:
A social-group appeal occurs when a social group is used as part of an argument rather than merely mentioned.

Assign yes when a social group is:
- affected by a problem,
- causing a problem,
- deserving praise, blame, protection, or support,
- benefiting from or harmed by a policy,
- the target of a proposed action.

Examples:
- "Immigrants are struggling." -> yes
- "Working mothers deserve support." -> yes
- "Immigrants live in Germany." -> no

DETECTED group:
{group}

RULES:
- Annotate only the DETECTED group in the TARGET sentence.
- Use PREV and NEXT only to understand the TARGET sentence.
- Replace <your_answer> with either "yes" or "no".

{item}

Return JSON:

{{
    "group_appealed": "<your_answer>"
}}
""".strip()

CLASSIFY_APPEAL_PROMPT = """
TASK:
Decide how the DETECTED group is appealed in the TARGET sentence.

DETECTED group:
{group}

{item}

## Annotation categories

### Reasoning

What is the primary basis of the argument about the group?

* economic: resources, jobs, wages, skills, productivity, taxes, welfare, public spending, economic incentives, economic performance
* cultural: values, norms, identity, religion, language, traditions, belonging, social cohesion, national identity
* both: both economic and cultural reasoning are central to the argument
* unclear: insufficient information to determine the primary basis

Examples:

* "Immigration depresses wages." → economic
* "Immigration threatens our national identity." → cultural
* "Immigration depresses wages and undermines our way of life." → both

### Polarity

How is the group represented?

* pro-group: deserving support, protection, inclusion, resources, rights, recognition, or improved outcomes
* anti-group: deserving criticism, blame, exclusion, restrictions, reduced support, or worse outcomes
* mixed: both positive and negative evaluations are present and neither dominates

Examples:

* "We should support disadvantaged children." → pro-group
* "Benefits for the unemployed should be reduced." → anti-group
* "We should support unemployed people, but benefits should be more conditional." → mixed

### Stance

What is the speaker's position toward the group appeal?

* endorse: accepts and relies on the group-based reasoning
* reject: disputes or invalidates the group-based reasoning
* partial: partially accepts but limits, reframes, or contests the appeal
* unclear: speaker's stance cannot be determined

Examples:

* "Working families are struggling, so they deserve tax relief." → endorse
* "It is not immigrants causing housing shortages." → reject
* "Immigration may affect wages, but it is not the main driver." → partial

## Instructions

1. Consider only the first appealed social group.
2. Use the surrounding sentences only as context.
3. Annotate only the target sentence.
4. Select exactly one value for each category.
5. Return valid JSON only.

Return JSON:

{{
  "reasoning": "economic|cultural|both|unclear",
  "polarity": "pro-group|anti-group|mixed",
  "stance": "endorse|reject|partial|unclear"
}}
""".strip()



In [36]:
# implement the dicision hierarchy and default outcome
from textwrap import fill

def fail(step, sent_id, text):
    print(f"\nFAILED at {step}")
    print(f"sent_id: {sent_id}")
    print(fill(text, width=80))

def safe_run_prompt(prompt):

    try:
        response = ask_llm(prompt)
        data = parse_json(response)
        return data

    except Exception as e:
        print(e)

        return None

def default_result():
    return {
        "groups": [],
        "group_mentioned": "no",
        "multiple_groups": "NA",
        "first_group": "",
        "group_appealed": "NA",
        "opposed_groups": "NA",
        "pejorative": "NA",
        "group_tags": ["NA"],
        "intersectional": "NA",
        "reasoning": "NA",
        "polarity": "NA",
        "stance": "NA",
    }

def annotate_item(sent_id, text, left_context="", right_context="",
                  verbose=True):

    result = default_result()

    context = f"""
PREV:
{left_context}

TARGET:
{text}

NEXT:
{right_context}
"""
    # step 1. extract groups
    prompt = MENTIONS_PROMPT.format(item=text)
    # response = ask_llm(prompt)
    # data = parse_json(response)
    data = safe_run_prompt(prompt)
    # [] is not None but a legit outcome
    if data is None:
        fail("mentions/group spans?", sent_id, text)
        result["groups"] = "FAILED"
        return result

    if verbose:
        print("\n=== MENTIONS RESULT ===")
        print(data)
        print("=" * 80 + "\n")

    groups = data["groups"]
    # store outcomes
    result["groups"] = groups

    # get the results for group_mentioned from the output
    result["group_mentioned"] = ("yes" if groups else "no")

    # early exit 1
    if not groups:
        if verbose:
            print("\nTARGET SENTENCE")
            print("-" * 80)
            print(fill(row["text"], width=80))
            print("No groups found. Early exit 1.\n")
        # return the default dictionary with "NA" for the rest of the fields
        return result

    first_group = groups[0]

    # store the span for the group we are annotating
    result["first_group"] = first_group

    # The previous stage result are inserted into the next prompt.
    # step 2. decide if the DETECTED group is appealed to
    data = safe_run_prompt(APPEAL_PROMPT.format(item=context, group=first_group))
    if data is None:
        fail("appealed?", sent_id, text)
        result["group_appealed"] = "FAILED"
        return result

    # store yes/no result for cases where groups are mentioned
    appeal_detected = data["group_appealed"]
    result["group_appealed"] = appeal_detected

    # early exit 2
    if appeal_detected == "no":
        if verbose:
            print("\nTARGET SENTENCE")
            print("-" * 80)
            print(fill(row["text"], width=80))
            print("No appealed groups found. Early exit 2.\n")
        # for the rest of the fields return pre-set defaults, i.e. "NA"
        return result

    # store the results for multiple_groups from the output
    result["multiple_groups"] = ("yes" if len(groups) > 1 else "no")

    # step 3. check if there are opposed groups
    data = safe_run_prompt(OPPOSED_PROMPT.format(item=context, groups=groups))

    if data is None:
        fail("opposed?", sent_id, text)
        result["opposed_groups"] = "FAILED"
        return result

    result["opposed_groups"] = data["opposed_groups"]

    # step 4. pejorative reference to first group which is detected as appealed?
    data = safe_run_prompt(PEJORATIVE_PROMPT.format(item=context, group=first_group))

    if data is None:
        fail("pejorative?", sent_id, text)
        result["pejorative"] = "FAILED"

    result["pejorative"] = data["pejorative"]

    # step 5. classify the first group: multiclass for each category
    data = safe_run_prompt(TAXONOMY_PROMPT.format(item=context, group=first_group))
    if data is None:
        fail("taxonomy?", sent_id, text)
        result["group_tags"] = "FAILED"
        return result

    result["group_tags"] = data["group_tags"]

    if len(data["group_tags"]) >= 2:
        result['intersectional'] = "yes"

    # step 6. run three multiclass classifications
    data = safe_run_prompt(CLASSIFY_APPEAL_PROMPT.format(item=context, group=first_group))
    # print(data)
    if data is None:
        fail("classify appeal?", sent_id, text)
        result["reasoning"] = "FAILED"
        result["polarity"] = "FAILED"
        result["stance"] = "FAILED"
        return result

    result["reasoning"] = data["reasoning"]
    result["polarity"] = data["polarity"]
    result["stance"] = data["stance"]

    return result

In [ ]:
# run on one example
from ast import literal_eval
import copy
import time
start = time.time()

ann = annotate_item(my_row["sent_id"],
                    my_row["text"],
                    my_row["left_context"],
                    my_row["right_context"],
                    verbose=True)
print(ann)

item_preds_df = pd.DataFrame([ann])

# post-process appeal tags
LABEL_MAPPING = {
    "reasoning": {
        "economic": "REASONING_ECONOMIC",
        "cultural": "REASONING_CULTURAL",
        "security": "REASONING_SECURITY",
        "unclear": "REASONING_UNCLEAR",
    },
    "polarity": {
        "pro-group": "POLARITY_PRO_GROUP",
        "anti-group": "POLARITY_ANTI_GROUP",
        "mixed": "POLARITY_MIXED",
        "unclear": "POLARITY_UNCLEAR"
    },
    "stance": {
        "endorse": "STANCE_ENDORSE",
        "reject": "STANCE_REJECT",
        "partial": "STANCE_PARTIAL",
        "unclear": "STANCE_UNCLEAR",
    }
}

# attach identifiers
item_preds_df.insert(0, "sent_id", sent_id)
# insert LLM's name as annotator
llm_annotator = model_name.split('/')[-1].lower()
item_preds_df.insert(0, "annotator", llm_annotator)

item_preds_df = item_preds_df.replace(LABEL_MAPPING)

print("\n" + "=" * 80)
print(item_preds_df)
print("=" * 80 + "\n")

end = time.time()
elapsed_time = (end - start) / 60
if elapsed_time < 60:
    print(f"\nTotal time: {elapsed_time:.1f} min")
else:
    print(f"\nTotal time: {int(elapsed_time // 60)} h {int(elapsed_time % 60)} min")


=== MENTIONS RESULT ===
{'groups': ['music teachers']}



In [ ]:
# compare to gold with the naked eye
from textwrap import fill

print("TARGET SENTENCE")
print("-" * 80)
print(fill(my_row["text"], width=80))

# post-process appeal tags
LABEL_MAPPING = {
    "reasoning": {
        "economic": "REASONING_ECONOMIC",
        "cultural": "REASONING_CULTURAL",
        "security": "REASONING_SECURITY",
        "unclear": "REASONING_UNCLEAR",
    },
    "polarity": {
        "pro-group": "POLARITY_PRO_GROUP",
        "anti-group": "POLARITY_ANTI_GROUP",
        "mixed": "POLARITY_MIXED",
        "unclear": "POLARITY_UNCLEAR"
    },
    "stance": {
        "endorse": "STANCE_ENDORSE",
        "reject": "STANCE_REJECT",
        "partial": "STANCE_PARTIAL",
        "unclear": "STANCE_UNCLEAR",
    }
}

item_preds_df = item_preds_df.replace(LABEL_MAPPING)

pred_row = item_preds_df.iloc[0]

print()
print(f'Detected group spans: {pred_row["groups"]}')
print()

print()
print(f'Group tags assigned to {pred_row["first_group"]}: {pred_row["group_tags"]}')
print()

print(f"{'Variable':<20} {'Pred':<10} {'Gold':<10} {'✓'}")
print("-" * 46)


for col in ["group_mentioned", "group_appealed", "multiple_groups",
            "intersectional", "opposed_groups", "pejorative",
            "reasoning", "polarity", "stance"]:
    ok = my_row[col] == pred_row[col]
    try:
      print(
          f"{col:<20} "
          f"{str(pred_row[col]):<10} "
          f"{str(my_row[col]):<10} "
          f"{'✓' if ok else '✗'}"
      )
    except ValueError:
      print(col)
      continue

# == Scaling to all available ajudicated items ==

In [ ]:
def postprocess(llm_preds, sent_ids):
    llm_preds = llm_preds.copy()

    LABEL_MAPPING = {
        "reasoning": {
            "economic": "REASONING_ECONOMIC",
            "cultural": "REASONING_CULTURAL",
            "security": "REASONING_SECURITY",
            "unclear": "REASONING_UNCLEAR",
        },
        "polarity": {
            "pro-group": "POLARITY_PRO_GROUP",
            "anti-group": "POLARITY_ANTI_GROUP",
            "mixed": "POLARITY_MIXED",
            "unclear": "POLARITY_UNCLEAR",
        },
        "stance": {
            "endorse": "STANCE_ENDORSE",
            "reject": "STANCE_REJECT",
            "partial": "STANCE_PARTIAL",
            "unclear": "STANCE_UNCLEAR",
        },
    }

    llm_preds.insert(0, "sent_id", sent_ids.to_numpy())
    # insert LLM's name as annotator
    llm_annotator = model_name.split('/')[-1].lower()
    llm_preds.insert(0, "annotator", llm_annotator)
    llm_preds.insert(3, "updated_at", date.today().isoformat())

    llm_preds = llm_preds.replace(LABEL_MAPPING)

    return llm_preds

In [ ]:
print(df.head())

In [ ]:
# run on all items
from datetime import date
import copy
import time

start = time.time()

results = []

for _, row in df.iterrows():

    my_ann = annotate_item(
        row["sent_id"],
        row["text"],
        row["left_context"],
        row["right_context"],
        verbose=False,
    )
    # print(my_ann)
    # print()

    results.append(my_ann)

sample_preds_df = pd.DataFrame(results)
print()
print(sample_preds_df.head())

end = time.time()
elapsed_time = (end - start) / 60
if elapsed_time < 60:
    print(f"\nTotal time: {elapsed_time:.1f} min")
else:
    print(f"\nTotal time: {int(elapsed_time // 60)} h {int(elapsed_time % 60)} min")

In [ ]:
sample_preds_df = postprocess(sample_preds_df, df["sent_id"])

print(set(df["sent_id"]) == set(sample_preds_df["sent_id"]))
print(len(set(sample_preds_df["sent_id"])))

print("\n" + "=" * 80)
print(sample_preds_df.head())
print("=" * 80 + "\n")

In [ ]:
import numpy as np
import pandas as pd

# combine gold and llm-generated labels and confirm the saniity of the format

gold_llm = pd.concat([df, sample_preds_df], axis=0)

print(gold_llm["annotator"].value_counts(dropna=False))

print(f"Empty values: {gold_llm["annotator"].isna().sum()}")

# # order gold and llm annotations pairwise
gold_llm["annotator"] = pd.Categorical(
    gold_llm["annotator"],
    categories=["gold", "qwen3-1.7b"],
    ordered=True,
)

gold_llm = gold_llm.sort_values(["sent_id", "annotator"]).reset_index(drop=True)

print(gold_llm[['annotator', 'sent_id', "group_mentioned", 'group_appealed', 'groups']].head())

In [ ]:
# re-use the same function that was used for calculating iaa between 3 humans
# you can import it, but I copy it in full below for debugging
import sys
sys.path.append("/content/DLDH-SocSci")
from iaa_and_gold import iaa_binary_multiclass, iaa_multilabel_categories

print(sys.path[-1])

# calculate IAA true global binary: group_mentioned
iaa_res_mention = iaa_binary_multiclass(gold_llm, ["group_mentioned"])

# calculate IAA where group_mentioned == "yes" in gold and llm annotations
gold_llm1 = gold_llm.groupby("sent_id").filter(lambda x: (x["group_mentioned"] == "yes").all())

# now group_appealed must be collapsed to binary and include only yes/no
print(gold_llm1["group_appealed"].value_counts())
print(gold_llm1[(gold_llm1["group_appealed"] == "NA") & (gold_llm1["group_mentioned"] == "yes")])

iaa_res_appealed = iaa_binary_multiclass(gold_llm1, ["group_appealed"])

# keep sent_ids where group_appealed == "yes" in all annotations
gold_llm2 = gold_llm1.groupby("sent_id").filter(lambda x: (x["group_appealed"] == "yes").all())

class_vars = ['intersectional', 'multiple_groups', 'opposed_groups', 'pejorative',
              'reasoning', 'polarity', 'stance']

iaa_res_multi = iaa_binary_multiclass(gold_llm2, class_vars)
iaa_res_bin = pd.concat([iaa_res_mention, iaa_res_appealed], ignore_index=True)
print(iaa_res_bin)

In [ ]:
cat_prefixes = ["AGE_", "GEN_", "FAM_", "DIS_", "ETH_", "MIG_", "CIT_",
                        "REL_", "GEO_", "EDU_", "INC_", "OCC_", "OTH_"]
cat_iaa_res = iaa_multilabel_categories(gold_llm2, "group_tags", cat_pref=cat_prefixes, meth="library")

In [ ]:
from iaa_and_gold import detailed_summary

detailed = detailed_summary(iaa_res_bin, iaa_res_multi, cat_iaa_res)
print(detailed)

In [ ]:
# separately look at IAA per-tag in group-tags
from iaa_and_gold import iaa_multilabel_tags
interface_df = pd.read_csv(REPO_DIR / "interface" / RUN / "interface_scheme.tsv", sep='\t')
interface_df = interface_df[~interface_df["tag"].str.contains("_*", regex=False, na=False)]

tag_iaa_res = iaa_multilabel_tags(gold_llm2, "group_tags", interface_df, min_freq=5)

# for per-tag view: keep only rows with valid alpha
tag_iaa_res_valid = tag_iaa_res.dropna(subset=["alpha"]).reset_index(drop=True)
# count rows with missing alpha
n_dropped = tag_iaa_res["alpha"].isna().sum()
print(f"\nPer-tag IAA is not defined for {n_dropped} (of {len(tag_iaa_res)})")
tag_iaa_res_valid = tag_iaa_res_valid.drop(["status"], axis=1)
# aggregate: add Avg row and round as before
avg_row = pd.DataFrame({
    "variable_type": ["Average"],
    "category": [""],
    "variable": [""],
    "agree_pct": [tag_iaa_res_valid["agree_pct"].mean().round(2)],
    "alpha": [tag_iaa_res_valid["alpha"].mean().round(3)],
    "tag_freq": [tag_iaa_res_valid["tag_freq"].mean().round(1)],
    "n_items": ["--"],
    "n_values": ["--"],
})

tag_valid_out = pd.concat([tag_iaa_res_valid, avg_row], ignore_index=True)
# tag_valid_out = tag_valid_out.astype({"n_items": "Int64", "n_values": "Int64"})
print(f"\nPer-tag IAA results (valid alpha) for {len(tag_iaa_res_valid)} tags:")
print(tag_valid_out.to_string(index=False))


In [ ]:
# anything stored under `/content/` is temporary and disappears when the Colab session ends.
from pathlib import Path
from google.colab import files

OUT_DIR = Path(REPO_DIR / "results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

sample_preds_df.to_csv(OUT_DIR / f"sample_preds_df_zero_{llm_annotator}.tsv", sep="\t", index=False)
detailed.to_csv(OUT_DIR / f"gold_{llm_annotator}_iaa_res.tsv", sep="\t", index=False)

tag_valid_out.to_csv(OUT_DIR / f"gold_{llm_annotator}_iaa_tag_level_social_groups_valid.tsv", sep="\t", index=False)

# files.download(OUT_DIR / "sample_preds_df.tsv")

# == End of zero-shot experimental branch ==
Feel free to copy the notebook and improve the results